# Phase 4 — Model Building
**Goal:** Train Logistic Regression, Random Forest and XGBoost. Compare performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, roc_auc_score, roc_curve)

X_train = joblib.load('../data/X_train.pkl')
X_test  = joblib.load('../data/X_test.pkl')
y_train = joblib.load('../data/y_train.pkl')
y_test  = joblib.load('../data/y_test.pkl')

print(f"Training: {X_train.shape} | Testing: {X_test.shape}")
print(f"Fraud in test: {y_test.sum()} out of {len(y_test)}")

In [ ]:
# Evaluation helper function
def evaluate_model(name, model, X_test, y_test):
    y_pred      = model.predict(X_test)
    y_pred_prob = model.predict_proba(X_test)[:,1]
    acc       = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall    = recall_score(y_test, y_pred)
    f1        = f1_score(y_test, y_pred)
    roc_auc   = roc_auc_score(y_test, y_pred_prob)

    print(f"\n{'='*50}")
    print(f"  MODEL: {name}")
    print(f"{'='*50}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}  ← most important for fraud")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  ROC-AUC   : {roc_auc:.4f}")
    print(f"{'='*50}")

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal','Fraud'], yticklabels=['Normal','Fraud'])
    plt.title(f'Confusion Matrix — {name}')
    plt.ylabel('Actual'); plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f'../reports/phase4_confusion_{name.replace(" ","_")}.png', dpi=150)
    plt.show()

    return {'Model':name,'Accuracy':round(acc,4),'Precision':round(precision,4),
            'Recall':round(recall,4),'F1':round(f1,4),'ROC-AUC':round(roc_auc,4)}

In [ ]:
# Model 1 — Logistic Regression (Baseline)
print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model.fit(X_train, y_train)
print("Done!")
lr_results = evaluate_model("Logistic Regression", lr_model, X_test, y_test)

In [ ]:
# Model 2 — Random Forest
print("Training Random Forest (1-2 mins)...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10,
                                   class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
print("Done!")
rf_results = evaluate_model("Random Forest", rf_model, X_test, y_test)

In [ ]:
# Model 3 — XGBoost (Champion)
fraud_count  = int(y_train.sum())
normal_count = int(len(y_train) - fraud_count)
scale_weight = normal_count / fraud_count
print(f"Training XGBoost... scale_pos_weight={scale_weight:.2f}")

xgb_model = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                            scale_pos_weight=scale_weight, eval_metric='logloss',
                            random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train, eval_set=[(X_test,y_test)], verbose=50)
print("Done!")
xgb_results = evaluate_model("XGBoost", xgb_model, X_test, y_test)

In [ ]:
# Model comparison table
results_df = pd.DataFrame([lr_results, rf_results, xgb_results]).set_index('Model')
print("\n=== MODEL COMPARISON ===")
print(results_df.to_string())
print("\nBest Recall:", results_df['Recall'].idxmax(),
      f"({results_df['Recall'].max():.4f})")

In [ ]:
# Comparison bar chart
metrics = ['Accuracy','Precision','Recall','F1','ROC-AUC']
x = np.arange(len(metrics)); width = 0.25
fig, ax = plt.subplots(figsize=(13,6))
ax.bar(x-width,   results_df.loc['Logistic Regression',metrics], width, label='Logistic Regression', color='steelblue',  alpha=0.85)
ax.bar(x,         results_df.loc['Random Forest',       metrics], width, label='Random Forest',        color='seagreen',  alpha=0.85)
ax.bar(x+width,   results_df.loc['XGBoost',             metrics], width, label='XGBoost',              color='crimson',   alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylim(0,1.1); ax.set_ylabel('Score')
ax.set_title('Model Comparison — LR vs RF vs XGBoost')
ax.legend(); ax.axhline(y=0.9, color='gray', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('../reports/phase4_model_comparison.png', dpi=150)
plt.show()

In [ ]:
# Feature importance
import pandas as pd
X_test_df = pd.DataFrame(X_test) if not hasattr(X_test,'columns') else X_test
feat_names = X_test_df.columns.tolist() if hasattr(X_test,'columns') else [f'f{i}' for i in range(X_test.shape[1])]
importances = pd.Series(xgb_model.feature_importances_, index=feat_names).sort_values(ascending=False).head(15)
plt.figure(figsize=(10,6))
importances.plot(kind='bar', color='crimson', edgecolor='black', alpha=0.85)
plt.title('Top 15 Feature Importances — XGBoost')
plt.ylabel('Importance Score'); plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../reports/phase4_feature_importance.png', dpi=150)
plt.show()

In [ ]:
# ROC Curve
plt.figure(figsize=(8,6))
for model, name, color in [(lr_model,'Logistic Regression','steelblue'),
                             (rf_model,'Random Forest','seagreen'),
                             (xgb_model,'XGBoost','crimson')]:
    prob = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', lw=2)
plt.plot([0,1],[0,1],'k--',label='Random baseline',lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — All Models'); plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../reports/phase4_roc_curve.png', dpi=150)
plt.show()

In [ ]:
# Save all models
joblib.dump(xgb_model, '../models/xgboost_model.pkl')
joblib.dump(lr_model,  '../models/logistic_model.pkl')
joblib.dump(rf_model,  '../models/random_forest_model.pkl')
print("All models saved!")
print("XGBoost is champion — feeds into Phase 5 Agentic AI")
print("Phase 4 complete!")